# Chargement des données
https://snap.stanford.edu/data/web-Google.txt.gz \
Données présentes en locale dans le dossier du script

In [57]:
import gzip
import pandas as pd # gestion des dataframe
import time # mesure du temps d'execution
import os, sys
import math

In [1]:
# ouverture du fichier d'origine avec les connexions existantes
with gzip.open('web-Google.txt.gz', 'rt') as f:
    data = f.read()
# Split des données sur la base du retour à la ligne
data = data.split("\n")
print("Nb de lignes dans le fichier original : ", len(data))
print(data[0:10])
print(data[ len(data)-5: ])

Nb de lignes dans le fichier original :  5105044
['# Directed graph (each unordered pair of nodes is saved once): web-Google.txt ', '# Webgraph from the Google programming contest, 2002', '# Nodes: 875713 Edges: 5105039', '# FromNodeId\tToNodeId', '0\t11342', '0\t824020', '0\t867923', '0\t891835', '11342\t0', '11342\t27469']
['916425\t604833', '916425\t632916', '916425\t637936', '916425\t837379', '']


Les quatres premières lignes sont commentées, elles correspondent à des indications de la source et la structure des données. \
Chaque ligne du fichier correspond à une arrête entre deux noeuds, pour un total de 875 713 Noeuds et 5 105 039 arrêtes. \
Le dernier élément de la liste est vide et doit être supprimé

In [2]:
# retrait des premières lignes et dernière qui ne comporte pas d'information de connexion entre 2 éléments # moins 4 lignes
while data[0][0] == "#" :
  data = data[1:]
while data[-1] == "" :
  data = data[:-1]
print("Nb de lignes avec connexion entre 2 éléments : ", len(data))
print(data[0:10])
print(data[len(data)-6:])

Nb de lignes avec connexion entre 2 éléments :  5105039
['0\t11342', '0\t824020', '0\t867923', '0\t891835', '11342\t0', '11342\t27469', '11342\t38716', '11342\t309564', '11342\t322178', '11342\t387543']
['916425\t521152', '916425\t547323', '916425\t604833', '916425\t632916', '916425\t637936', '916425\t837379']


Pour comparer les différentes approches, création d'une table pour enregistrer les résultats de test et définition des variables à tester.

In [4]:
df_comp = pd.DataFrame(columns=["methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps"])
nb_parallele = [1,2,4,6]
nb_edges = [10 , 100, 1000, 10000, 100000 , 500000 , 1000000, 2000000] # Erreur avec len(data)?
nb_test = 3

# 1er méthode : traitement sur dataframe en python

In [4]:
for lignes in nb_edges[0:5] : # test différentes tailles de fichier
    for i in range(nb_test) : # répétition du test pour la mesure du temps d'éxécution
        start = time.time()
    
        # Création du dataframe selon la taille souhaitée
        df = pd.DataFrame([x.split("\t") for x in data[0:lignes]], columns=["FromNodeId", "ToNodeId"])
    
        cptnewpair = 1 # compteur pour vérifier la sortie du while
        cpt = 0 # compteur nombre itération
            
        while cptnewpair >= 1 : 
        
            ##########################################
            ### Map génération des couples de lien ###
            ##########################################
            
            # inversion des deux colonnes pour générer le couple dans l'autre sens 
            forward = df 
            reverse = df[["ToNodeId", "FromNodeId"]].rename(columns={"ToNodeId": "FromNodeId", "FromNodeId": "ToNodeId"}) 
            # merge des résultats 
            df = pd.concat([forward, reverse])
            # suppression des doublons 
            df = df.drop_duplicates().reset_index(drop=True)
            
            #############################################################
            ### Equivalent étape shuffle en sort regroupement par clé ###
            #############################################################
            
            dictionnaire = {}
            for l in range (df.shape[0]) : 
                cle = df.at[l, "FromNodeId"]
                valeur = df.at[l, "ToNodeId"]
                if cle not in dictionnaire :
                    dictionnaire[cle] = [valeur]
                else:
                    dictionnaire[cle].append(valeur)
            
            ##############################################################
            ### Reduce génération du nouveau data frame avec les liens ###
            ##############################################################
            
            # Initialisation de la table sortie data dans dictionnaire
            df = pd.DataFrame(columns=("FromNodeId", "ToNodeId"))
            cptnewpair = 0
            
            for key, values in dictionnaire.items() : 
                mini = key 
                for v in values : 
                    if v < mini : 
                        mini = v 
                if mini < key : 
                    df.loc[len(df)] = [ key , mini ]
                    for v in values : 
                        if mini != v : 
                            cptnewpair += 1
                            df.loc[len(df)] = [v , mini ]
    
            cpt += 1 
    
        # Ajout des informations dans un dataframe
        temps = round(time.time() - start,3)
        nb_arrete = df.shape[0] # mesure du nombre d'arrêtes restantes après l'algorithme CCF
        nb_groupe = df["ToNodeId"].nunique()
        
    
        df_comp.loc[len(df_comp)] = ["python_df", lignes, nb_arrete, nb_groupe, cpt, 1,  temps ]          

In [5]:
# df_comp.to_csv("Comp_python.csv", index=False)

# Utilisation de spark en python 
https://spark.apache.org/docs/latest/rdd-programming-guide \
https://spark.apache.org/examples.html 

Analyse des paramètres machine

In [37]:
import psutil
import multiprocessing

# Nombre de cœurs
print("--- Info Machine ---")
print("Cœurs logiques :", multiprocessing.cpu_count())
print("Cœurs physiques :", psutil.cpu_count(logical=False))
# RAM
mem = psutil.virtual_memory()
print("Mémoire totale (Go):", round(mem.total / (1024**3), 2))
print("Mémoire disponible (Go):", round(mem.available / (1024**3), 2))

# Infos Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext
print("\n--- Info Spark ---")
print("Spark master :", sc.master)
print("Spark parallelism :", sc.defaultParallelism)
print("Driver memory (si défini):", sc.getConf().get("spark.driver.memory", "non défini"))

--- Info Machine ---
Cœurs logiques : 8
Cœurs physiques : 4
Mémoire totale (Go): 3.77
Mémoire disponible (Go): 1.56

--- Info Spark ---
Spark master : local[6]
Spark parallelism : 6
Driver memory (si défini): 1g


Chargement des librairies et démarrage du spark contexte pour accéder aux RDD

In [5]:
# # Force Spark à utiliser le Python de l'env conda
# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# # install spark sur la machine colab
# !pip install pyspark
# Chargement des paquets
from pyspark import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
import pyspark.pandas as ps

# Commande, ici très simple car une seule machine
# Création d'un objet sc, class spark context, à partir de sc possibilité de faire les RDD
spark = SparkSession.builder \
    .appName("projet") \
    .master("local[6]") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "3") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = SparkContext.getOrCreate(SparkConf().setMaster("local[6]")) # local et * tout les coeurs

# 2nd méthode : spark python sur dataframe
https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/frame.html \
Gestion dataframe avec spark non optimum, passage via sql plus optimisé

In [7]:
# # Création du dataframe selon la taille souhaitée
# df = ps.DataFrame([x.split("\t") for x in data[0:lignes]], columns=["FromNodeId", "ToNodeId"])
# reverse = df[["ToNodeId", "FromNodeId"]].rename(columns={"ToNodeId": "FromNodeId", "FromNodeId": "ToNodeId"})
# df = ps.concat([forward, reverse])
# df = df.drop_duplicates().reset_index(drop=True) # supprim les doublons et tri équivalent groupby

# # Identification de la cle 
# cle = df["FromNodeId"].drop_duplicates().reset_index(drop = True)

# # Abandon

# 3ème méthode : spark python sur dataframe SQL
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/core_classes.html

In [60]:
# Import des librairies nécessaires
from pyspark.sql.functions import col, udf, explode, spark_partition_id, countDistinct
from pyspark.sql import functions
from pyspark.sql.types import IntegerType, ArrayType
import math

### fonction pour identifier la plus petite valeur dans la liste des "values" de la noeud2 après le shuffle and sort
# udf = user-defined functions
@udf(returnType="int")
def value_min(info ):
    mini = math.inf
    for n in info:  # par exemple : une ligne par nombre
        if n < mini : 
            mini = n 
    return mini

### Identification des valeurs qui seront à associée à la valeur mini sur le format (result, mini)
@udf(ArrayType(IntegerType()))
def ccf_iterate_df(info1, info2 , mini):
    results = []
    if mini < info1 : 
        results.append(info1) 
        for n in info2:  # par exemple : une ligne par nombre
            if n != mini : 
                results.append(n)
                # compteur.add(1) # augmentation du compteur non fonctionnel dans udf
    return results
# df.dtypes # vérification des type de colonnes 


### Identification des valeurs qui seront à associée à la valeur mini sur le format (result, mini)
@udf(ArrayType(IntegerType()))
def ccf_iterate_sort_df(info1, info2, mini):
    results = []
    if mini < info1 : 
        results.append(info1) 
        for n in info2[1:] :  # premier element non pris en compte car il s'agit de la valeur minimale 
            # if n != mini : par de paires en double donc sans doublon dans info2 inutile de tester n != mini
            results.append(n)
            # compteur.add(1) # augmentation du compteur non fonctionnel dans udf
    return results
# df.dtypes # vérification des type de colonnes 

In [63]:
def runCCF_DF( data, lignes, param_deduplic, param_algo , method, df_comp ) :
    
    # Création du sql.dataframe pyspark
    df = spark.createDataFrame( [tuple(map(int, s.split("\t"))) for s in data[0:lignes]],  ["noeud1", "noeud2"] ) 
    # compteur = sc.accumulator(1) # non fonctionnel avec udf 
    compteur = 1 
    cpt = 0 # compteur nombre itération
    
    start = time.time()
    
    while compteur >= 1 :            
        ####################################################
        # map, ajout de arêtes dans l'autre orientation ###
        ####################################################
        reverse = df.select(col("noeud2").alias("noeud1"), col("noeud1").alias("noeud2"))
        if param_deduplic == "yes":
            df = df.union(reverse).distinct() # merge des deux tables et suppression des doublons
        else : 
            df = df.union(reverse)

        ########################
        ### shuffle and sort ###
        ########################
        # Grouper sur la colonne 1 noeud1 = clé et créer la liste des valeurs de la seconde colonne = values
        df = df.groupBy("noeud1").agg(functions.collect_list("noeud2").alias("noeud2"))
        
        ###################
        ### ccf-iterate ###
        ###################

        if param_algo == "num1" :
            # recherche de la plus petites valeurs dans la colonne noeud2 avec la fonction value_min et export dans une nouvelle colonne
            df = df.withColumn("mini", value_min(df["noeud2"]))
            # Identification des noeuds au format liste associés à la valeur mini à conserver pour la prochaine itération ou le graph final
            df = df.withColumn("processed", ccf_iterate_df(df["noeud1"], df["noeud2"], df["mini"]))
        else : 
            # Second algo, tri des données agrégé pour extraire la valeur minimum
            # tri en place de la collection obtenue après le shuffle and sort dans la colonne "noeud2"
            df = df.withColumn("noeud2", functions.array_sort("noeud2"))
            # EXtraction du premier element en tant que mini, conservation de le même structure que l'autre condition
            df = df.withColumn("mini", functions.col("noeud2")[0])
            # Identification des noeuds au format liste associés à la valeur mini à conserver pour la prochaine itération ou le graph final
            df = df.withColumn("processed", ccf_iterate_sort_df(df["noeud1"], df["noeud2"], df["mini"]))
        
        # Etendre le résultat inclus dans la liste des exports de CCF
        df = df.select("*", functions.explode('processed'))

        #l'accumulateur ne fonctionne pas 
        # vérification si besoin de faire une itération de plus si présence du même noeud dans les deux colonnes
        # compteur = df.select("mini").distinct().join(df.select("col").distinct()).\
        compteur =  df.filter(functions.col("mini") == functions.col("col")).count()
        
        # reformatage du résultat dans le format initial
        df = df.select(df.col.alias("noeud1"),df.mini.alias("noeud2")).distinct()

        cpt += 1 
               
    temps = round(time.time() - start,3)
    nb_parallele = df.withColumn("partitionid", spark_partition_id()).select("partitionid").\
                    agg(countDistinct("partitionid")).count()

    # Ajout des informations dans un dataframe
    nb_arete = df.count() # mesure du nombre d'arrêtes restantes après l'algorithme CCF
    nb_groupe = df.select("noeud2").distinct().count()
    
    # Ajout des informations dans un dataframe suivi performances
    # df_comp.loc[len(df_comp)] = [method, lignes, nb_arete, nb_groupe, cpt, nb_parallele,  temps ]

    return [method, lignes, nb_arete, nb_groupe, cpt, nb_parallele,  temps ]

In [64]:
import math 
df_comp = pd.DataFrame(columns=["methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps"])

nb_edges = [10, 100, 1000, 10000, 100000]*3 # taille du dataset et nb de répétition

for lignes in nb_edges : # test différentes tailles de fichier
    # def runCCF_DF( data, lignes, param_duplic, param_algo , method, df_comp )      
    df_comp.loc[len(df_comp)]  = runCCF_DF( data, lignes, "no", "num1" , "df_python_iterate_avec_duplicat", df_comp )
    df_comp.loc[len(df_comp)]  = runCCF_DF( data, lignes, "yes", "num1" , "df_python_iterate_sans_duplicat", df_comp )
    df_comp.loc[len(df_comp)]  = runCCF_DF( data, lignes, "yes", "num2" , "df_python_iterate_sort_sans_duplicat", df_comp )

df_comp.to_csv("/data/indir/Comp_python_DF_add_config_spark.csv", index=False)
df_comp.head()

,methode,nb_init_arretes,nb_final_arretes,nb_groupes,nb_iter_CCF,nb_parellele,temps
0,df_python_iterate_avec_duplicat,10,14,2,1,1,2.145
1,df_python_iterate_sans_duplicat,10,14,2,1,1,3.274
2,df_python_iterate_sort_sans_duplicat,10,14,2,1,1,1.931
3,df_python_iterate_avec_duplicat,100,104,5,1,1,1.914
4,df_python_iterate_sans_duplicat,100,104,5,1,1,1.293


In [65]:
# df_comp.to_csv("/data/indir/Comp_python_DF_add_config_spark.csv", index=False)

# 4ème méthode : spark python sur RDD
https://spark.apache.org/docs/latest/api/python/reference/pyspark.html

### Définition des fonctions CCF
En entrée : 

- x = est un couple de clé/valeur qui correspond à deux noeuds pour former une arête

- y = est un couple de clé/collection de valeur

In [42]:
# En entrée x est un couple de clé/valeur qui correspond à deux noeuds pour former une arête
# ccf_iterate_map permet de créer l'arrête avec la liaison dans les deux orientations
def ccf_iterate_map_rdd (x) :
    return [(x[0],x[1]),(x[1],x[0])]

# En entrée x est un couple de clé/valeurs, la clé est unique et valeurs peut-être unique ou mulitple
# il s'agit de différentes connexion avec la clé
def ccf_iterate_rdd (y, compteur=None) :
    l = [] # initialisation des données de sortie
    key = y[0] # stockage de la clé
    min = y[0] # initialisation de la valeur la plus faible
    # recherche dans l'ensemble des valeurs si présence d'une valeur plus faible
    for val in y[1] :
        if val < min :
            min = val
    # si dans les valeurs présence d'une valeur plus faible que la clé
    if (min < key ) :
        l.append( (key, min) ) # sortie du couple clé/val_min
        for val in y[1] :
            if val != min :
                l.append( (val, min) ) # export des nouveaux couples
                compteur.add(1) # augmentation du compteur
    return l

# En entrée x est un couple de clé/valeurs, la clé est unique et valeurs peut-être unique ou mulitple
# il s'agit de différentes connexion avec la clé
def ccf_iterate_sort_rdd (y, compteur=None) : 
    l = [] # initialisation des données de sortie
    key = y[0] # stockage de la clé
    valeur = sorted(y[1]) 
    min = valeur[0]
    # si présence d'une valeur plus faible que la clé
    if ( min < key ) :
        l.append( (key, min) ) # sortie du couple clé/val_min
        for val in valeur[1:] :
            l.append( (val, min) ) # export des nouveaux couples
            compteur.add(1) # augmentation du compteur
    return l

In [43]:
def runCCF_RDD( data, taille_ech, ccfRDD , param_parall, param_deduplic, method, df_comp ) : 
    
    # Création du RDD et structuration des données
    data_rdd = sc.parallelize(data[0:taille_ech] , param_parall).map(lambda x : ( x.split("\t")[0] , x.split("\t")[1] ) )
    
    cpt = 0 # Initialisation du compteur de boucle

    start = time.time()
    while True :
        # Intialisation de l'accumulateur pour chaque itération
        compteur = sc.accumulator(0)
        
        # Transformations selon l'algo CCF
        # Transformation initiale création des paires bidirectionnelles
        data_rdd = data_rdd.flatMap(lambda x: ccf_iterate_map_rdd(x))
        
        # Optionnel : suppression des doublons
        if param_deduplic == "yes":
            data_rdd = data_rdd.distinct()
        
        # groupByKey + application de la fonction passée en argument
        data_rdd = data_rdd.groupByKey().flatMap(lambda x: ccfRDD(x, compteur))

        # Action pour déclencher le calcul et mettre à jour l'accumulateur
        data_rdd.count()
        # Actualisation du nombre d'itération et affichage
        cpt += 1
        # print(f"Iteration {cpt}, compteur = {compteur.value}")
        # condition d’arrêt
        if compteur.value == 0:
            break
    
    temps = round( time.time() - start,3)
    # Ajout des informations dans un dataframe
    nb_arrete = data_rdd.count() # mesure du nombre d'arrêtes restantes après l'algorithme CCF
    nb_groupe = data_rdd.map(lambda x : (x[1], x[0])).groupByKey().count() # identification du nombre de groupe
  
    df_comp.loc[len(df_comp)] = [method+"_"+str(param_parall), taille_ech, nb_arrete, nb_groupe, cpt, param_parall, temps ]
    
    return df_comp

In [45]:
df_comp = pd.DataFrame(columns=["methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps"])

nb_edges = [10, 100, 1000, 10000, 100000, 500000, 1000000]*3 # taille du dataset et nb de répétition
nb_parallele = [3]

for lignes in nb_edges : # test différentes tailles de fichier
    for parall in nb_parallele : 
            
        df_comp = runCCF_RDD( data, lignes, ccf_iterate_rdd , parall, "no", "RDD_python_iterate_avec_duplicat", df_comp ) 
        df_comp = runCCF_RDD( data, lignes, ccf_iterate_rdd , parall, "yes", "RDD_python_iterate_sans_duplicat", df_comp )
        df_comp = runCCF_RDD( data, lignes, ccf_iterate_sort_rdd , parall, "yes", "RDD_python_iterate_sort_sans_duplicat", df_comp )

df_comp.to_csv("/data/indir/Comp_python_RDD_add_config_spark.csv", index=False)
df_comp.head()

,methode,nb_init_arretes,nb_final_arretes,nb_groupes,nb_iter_CCF,nb_parellele,temps
0,RDD_python_iterate_avec_duplicat_3,10,9,1,4,3,1.200
1,RDD_python_iterate_sans_duplicat_3,10,9,1,4,3,1.790
2,RDD_python_iterate_sort_sans_duplicat_3,10,9,1,4,3,1.984
3,RDD_python_iterate_avec_duplicat_3,100,72,2,4,3,1.240
4,RDD_python_iterate_sans_duplicat_3,100,72,2,4,3,2.310


In [ ]:
# test de traitement du dataframe complet 

df_comp = pd.DataFrame(columns=["methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps"])

df_comp = runCCF_RDD( data, len(data) , ccf_iterate_rdd , parall, "yes", "RDD_python_iterate_sans_duplicat", df_comp )
df_comp.to_csv("/data/indir/Comp_python_5M_config_spark.csv", index=False)

df_comp = runCCF_RDD( data, lignes, ccf_iterate_sort_rdd , parall, "yes", "RDD_python_iterate_sort_sans_duplicat", df_comp )
df_comp.to_csv("/data/indir/Comp_python_5M_config_spark.csv", index=False)

df_comp.loc[len(df_comp)]  = runCCF_DF( data, len(data), "yes", "num1" , "df_python_iterate_sans_duplicat", df_comp )
df_comp.to_csv("/data/indir/Comp_python_5M_config_spark.csv", index=False)

df_comp.loc[len(df_comp)]  = runCCF_DF( data, len(data), "yes", "num2" , "df_python_iterate_sort_sans_duplicat", df_comp )
df_comp.to_csv("/data/indir/Comp_python_5M_config_spark.csv", index=False)

df_comp.head()

In [69]:
len(data)

5105039

In [46]:
df_comp.tail()

,methode,nb_init_arretes,nb_final_arretes,nb_groupes,nb_iter_CCF,nb_parellele,temps
40,RDD_python_iterate_sans_duplicat_3,10000,4129,59,8,3,5.055
41,RDD_python_iterate_sort_sans_duplicat_3,10000,4129,59,8,3,5.423
42,RDD_python_iterate_avec_duplicat_3,100000,39923,381,9,3,7.992
43,RDD_python_iterate_sans_duplicat_3,100000,39923,381,9,3,12.158
44,RDD_python_iterate_sort_sans_duplicat_3,100000,39923,381,9,3,12.124


In [14]:
df_comp

,methode,nb_init_arretes,nb_final_arretes,nb_groupes,nb_iter_CCF,nb_parellele,temps
0,Spark_python_RDD_6,10,9,1,4,6,4.287
1,Spark_python_RDD_6,10,9,1,4,6,2.076
2,Spark_python_RDD_6,10,9,1,4,6,2.252
3,Spark_python_RDD_6,100,72,2,4,6,2.381
4,Spark_python_RDD_6,100,72,2,4,6,1.929
5,Spark_python_RDD_6,100,72,2,4,6,2.051
6,Spark_python_RDD_6,1000,419,11,6,6,3.115
7,Spark_python_RDD_6,1000,419,11,6,6,2.892
8,Spark_python_RDD_6,1000,419,11,6,6,2.852
9,Spark_python_RDD_6,10000,4129,59,8,6,3.993


In [15]:
# len(data)
# data_rdd = sc.parallelize(data[0:lignes] , parall).map(lambda x : ( x.split("\t")[0] , x.split("\t")[1] ) )
 